<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/TextCleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from collections import Counter

input_file = "HM_Clean_00.txt"
output_file = "HM_wordcount.txt"

with open(input_file, "r", encoding="cp1252") as f:
#with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

words = re.findall(r"\b[A-Za-z]+\b", text)
counts = Counter(word.lower() for word in words)

with open(output_file, "w", encoding="utf-8") as f:
    for word in sorted(counts):
        if counts[word] >= 2:
            f.write(f"{word:<25} {counts[word]:>6}\n")

print(f"Done. {len(counts):,} unique words written to {output_file}")

Done. 5,904 unique words written to HM_wordcount.txt


In [ ]:
!pip -q install wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 13.1 MB/s eta 0:00:00


In [ ]:
import re
from collections import Counter
from wordfreq import zipf_frequency

input_file = "HM_Clean_00.txt"
output_file = "HM_special_wordcount.txt"

# Read Harihar
with open(input_file, "r", encoding="cp1252") as f:
    text = f.read()

# Extract words
words = re.findall(r"\b[A-Za-z]+\b", text)
counts = Counter(word.lower() for word in words)

with open(output_file, "w", encoding="utf-8") as f:

    for word in sorted(counts):

        # Ignore very rare words
        if counts[word] < 2:
            continue

        # Ignore single-letter OCR rubbish
        if len(word) < 2:
            continue

        # If wordfreq recognises it as a reasonably
        # common English word, throw it away
        if zipf_frequency(word, "en") >= 2.5:
            continue

        f.write(f"{word:<30} {counts[word]:>6}\n")

print("Done.")
print("Output:", output_file)

Done.
Output: HM_special_wordcount.txt


In [ ]:
word = "pusa"   # change this to the suspicious word

found = False

for lineno, line in enumerate(text.splitlines(), 1):
    if word.lower() in line.lower():
        print(f"Line {lineno}:")
        print(repr(line))
        found = True

if not found:
    print("NOT FOUND:", word)

Line 211:
"   Punarvasu Nakshatra: Spread from 20° degree Mithune upto 3°-20' Karkata. Presiding deity 'Aditi', the Lord is Budha and Chandra. Symbol - Quiver ( receptacle for arrows). The word Punarvasu is derived from Puna + Vasu, which means return, renewal, restoration or repetition. The 12 Adityas were born of Kasyapa in the womb of Aditi. The 12 are Indra, Vishnu, Vaga, Twasta, Barun, Aryama, Pusa., Mitra, Agni, Parjyanya, Vivaswan and Dinakar. The mother Aditi of whom the Gods are born is the repository of everything good - truth, generosity, magnanimity, purity, aristocracy, beauty and renown. It follows that this start is the cause for these virtues. To start afresh after having once broken off, to start a new life, to come back from a distant land - all these are signified by Punarvasu. It stands for freedom from restriction and limitation, and boundless space. The Gods. the children of Aditi, are basically and essentially are different from children of Diti, who are demons. 

In [ ]:
import re
import nltk
from collections import Counter
from nltk.corpus import words as nltk_words

# Download dictionary if necessary
nltk.download("words", quiet=True)

input_file = "HM_Clean_00.txt"
output_file = "HM_Special_Wordcount.txt"

# English dictionary
english_words = {w.lower() for w in nltk_words.words()}

# HM_Clean_00 is the clean starting corpus
# If this gives a UnicodeDecodeError, change utf-8 to cp1252
with open(input_file, "r", encoding="cp1252") as f:
    text = f.read()

# Extract alphabetic tokens only.
# Thus "Pusa." becomes "pusa", which is intentional.
tokens = re.findall(r"\b[A-Za-z]+\b", text)

# Case-insensitive frequency count
counts = Counter(token.lower() for token in tokens)

special = []

for word, count in counts.items():

    # Ignore words of 2 characters or less
    if len(word) <= 2:
        continue

    # Ignore words occurring only once
    if count < 2:
        continue

    # Ignore recognised English dictionary words
    if word in english_words:
        continue

    special.append((word, count))

# Alphabetical order
special.sort()

with open(output_file, "w", encoding="utf-8") as f:
    for word, count in special:
        f.write(f"{word:<30} {count:>6}\n")

print(f"Total tokens in corpus : {len(tokens):,}")
print(f"Unique tokens          : {len(counts):,}")
print(f"Special words retained : {len(special):,}")
print(f"Written to             : {output_file}")

Total tokens in corpus : 74,588
Unique tokens          : 5,904
Special words retained : 830
Written to             : HM_Special_Wordcount.txt


In [2]:
import re
import nltk
from collections import Counter
from nltk.corpus import words as nltk_words

# --------------------------------------------------
# HM SIEVE 01
# Find recurring non-English / unusual words
# --------------------------------------------------

nltk.download("words", quiet=True)

input_file = "HM_Clean_01.txt"
output_file = "HM_Sieve_01a.txt"

# English dictionary
english_words = {w.lower() for w in nltk_words.words()}

# Read corpus
# If utf-8 fails, change encoding to "cp1252"
with open(input_file, "r", encoding="cp1252") as f:
    text = f.read()

# Extract alphabetic words only and convert to lowercase
tokens = re.findall(r"\b[A-Za-z]+\b", text)
counts = Counter(word.lower() for word in tokens)

special = []

for word, count in counts.items():

    # Ignore very short tokens
    if len(word) <= 2:
        continue

    # Ignore one-off occurrences
    if count < 2:
        continue

    # Ignore recognised English words
    if word in english_words:
        continue

    special.append((word, count))

# Alphabetical order
special.sort()

with open(output_file, "w", encoding="utf-8") as f:
    for word, count in special:
        f.write(f"{word:<30} {count:>6}\n")

print(f"Total tokens          : {len(tokens):,}")
print(f"Unique tokens         : {len(counts):,}")
print(f"Sieve 01 candidates   : {len(special):,}")
print(f"Written to            : {output_file}")

Total tokens          : 74,490
Unique tokens         : 5,826
Sieve 01 candidates   : 788
Written to            : HM_Sieve_01a.txt


In [3]:
import re
from collections import Counter

# --------------------------------------------------
# HM SIEVE 02
# Find probable OCR errors by near-neighbour matching
# --------------------------------------------------

input_file = "HM_Clean_01.txt"
output_file = "HM_Sieve_02a.txt"

# Read corpus
# If utf-8 fails, change encoding to "cp1252"
with open(input_file, "r", encoding="cp1252") as f:
    text = f.read()

# Extract alphabetic words and normalise case
tokens = re.findall(r"\b[A-Za-z]+\b", text)
counts = Counter(word.lower() for word in tokens)


# --------------------------------------------------
# Levenshtein distance
# --------------------------------------------------

def edit_distance(a, b):

    # Quick rejection
    if abs(len(a) - len(b)) > 1:
        return 99

    previous = list(range(len(b) + 1))

    for i, ca in enumerate(a, start=1):

        current = [i]

        for j, cb in enumerate(b, start=1):

            insertion = current[j - 1] + 1
            deletion = previous[j] + 1
            substitution = previous[j - 1] + (ca != cb)

            current.append(min(insertion, deletion, substitution))

        previous = current

    return previous[-1]


# --------------------------------------------------
# Candidate selection
# --------------------------------------------------

# Suspect words:
# occur at least twice, but no more than 20 times
suspects = [
    word for word, count in counts.items()
    if len(word) > 2
    and 2 <= count <= 20
]

# Possible correct words:
# occur at least 10 times
reference_words = [
    word for word, count in counts.items()
    if len(word) > 2
    and count >= 10
]

results = []

for suspect in suspects:

    for reference in reference_words:

        if suspect == reference:
            continue

        # Only compare words of almost identical length
        if abs(len(suspect) - len(reference)) > 1:
            continue

        # Reference should be substantially more common
        if counts[reference] < counts[suspect] * 3:
            continue

        # Exactly one insertion/deletion/substitution away
        if edit_distance(suspect, reference) == 1:

            results.append(
                (
                    suspect,
                    counts[suspect],
                    reference,
                    counts[reference]
                )
            )


# Sort by suspect word, then strongest reference candidate
results.sort(key=lambda x: (x[0], -x[3]))


# --------------------------------------------------
# Write report
# --------------------------------------------------

with open(output_file, "w", encoding="utf-8") as f:

    f.write(
        f"{'SUSPECT':<25}"
        f"{'COUNT':>8}    "
        f"{'POSSIBLE CORRECTION':<25}"
        f"{'COUNT':>8}\n"
    )

    f.write("-" * 72 + "\n")

    for suspect, scount, reference, rcount in results:

        f.write(
            f"{suspect:<25}"
            f"{scount:>8}    "
            f"{reference:<25}"
            f"{rcount:>8}\n"
        )


print(f"Total tokens          : {len(tokens):,}")
print(f"Unique tokens         : {len(counts):,}")
print(f"Low-frequency suspects: {len(suspects):,}")
print(f"Sieve 02 candidate pairs: {len(results):,}")
print(f"Written to            : {output_file}")

Total tokens          : 74,490
Unique tokens         : 5,826
Low-frequency suspects: 2,540
Sieve 02 candidate pairs: 452
Written to            : HM_Sieve_02a.txt


In [1]:
import csv
import re

INPUT_FILE   = "HM_Clean_00.txt"
REPLACE_FILE = "replace2.txt"
OUTPUT_FILE  = "HM_Clean_01.txt"

# ------------------------------------------------------------
# 1. Load replacement table
#    Format: wrong,right,category
#    The category column is deliberately ignored.
# ------------------------------------------------------------

replacements = {}

with open(REPLACE_FILE, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)

    for row in reader:
        wrong = row["wrong"].strip()
        right = row["right"].strip()

        if wrong and right:
            replacements[wrong] = right

print(f"Loaded {len(replacements)} replacement rules.")


# ------------------------------------------------------------
# 2. Read HM_Clean_00
# ------------------------------------------------------------

with open(INPUT_FILE, "r", encoding="cp1252") as f:
    text = f.read()


# ------------------------------------------------------------
# 3. Replace whole words only
#    Case-insensitive, but preserve the replacement exactly
#    as specified in replace2.txt.
# ------------------------------------------------------------

total_replacements = 0

for wrong, right in replacements.items():

    pattern = r"\b" + re.escape(wrong) + r"\b"

    text, count = re.subn(
        pattern,
        right,
        text,
        flags=re.IGNORECASE
    )

    if count > 0:
        print(f"{wrong:20s} -> {right:20s} : {count}")

    total_replacements += count


# ------------------------------------------------------------
# 4. Write HM_Clean_01
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(text)


print("\n----------------------------------------")
print(f"Total replacements : {total_replacements}")
print(f"Output written to  : {OUTPUT_FILE}")
print("----------------------------------------")

Loaded 51 replacement rules.
krithika             -> krittika             : 7
poonarvasu           -> punarvasu            : 2
pusya                -> pushya               : 4
jestha               -> jyesta               : 3
dhanista             -> dhanistha            : 2
poorbasarha          -> purvashadha          : 7
uttarasarha          -> uttarashadha         : 7
poorbavadrapada      -> purvabhadra          : 2
purvabhadrapada      -> purvabhadra          : 3
uttarbhadra          -> uttarbhadrapada      : 1
rebati               -> revati               : 4
revathi              -> revati               : 1
visakha              -> visaka               : 8
budba                -> budha                : 3
mangat               -> mangal               : 9
rahn                 -> rahu                 : 4
vrbaspatbi           -> vrhaspati            : 3
vrbaspathi           -> vrhaspati            : 6
vrhaspatbi           -> vrhaspati            : 10
vrhaspathi           -> vrhaspati      